# Notebook 1.5: Upload Knowledge Base to S3

## Overview

In this notebook, we will:
1. Load the chunked knowledge base from Notebook 01
2. Create individual text files for each chunk
3. Upload files to S3 in organized folders
4. Generate a manifest file with metadata
5. Prepare for Bedrock Knowledge Base creation

**Learning Objectives:**
- Understand S3 bucket structure for Knowledge Bases
- Learn how to organize documents for vector indexing
- See the upload process and verification
- Prepare for ingestion into Bedrock

**Prerequisites:**
- Completed Notebook 01 (chunks generated)
- S3 bucket created in AWS
- AWS credentials configured in .env

---

## 1. Setup and Configuration

In [ ]:
import os
import json
import boto3
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configuration
BUCKET_NAME = "ecom-rag-bucket"  # Change this to your bucket name
AWS_REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')

# Create timestamped folder name (top-level, not nested)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
KB_FOLDER = f"knowledge-base-{timestamp}"  # Each upload gets its own folder

# Initialize S3 client
s3_client = boto3.client('s3', region_name=AWS_REGION)

print("[OK] Configuration loaded")
print(f"  Bucket: {BUCKET_NAME}")
print(f"  KB Folder: {KB_FOLDER}")
print(f"  Full S3 Path: s3://{BUCKET_NAME}/{KB_FOLDER}/")
print(f"  Region: {AWS_REGION}")
print()
print("This creates a NEW independent folder - no duplicate ingestion!")
print("Each Bedrock Knowledge Base data source points to one folder.")

---

## 2. Verify S3 Bucket Access

Check that we can access the S3 bucket.

# List existing knowledge base folders
print("Checking for existing knowledge base folders...")
print("-"*80)

try:
    response = s3_client.list_objects_v2(
        Bucket=BUCKET_NAME,
        Delimiter='/'
    )
    
    if 'CommonPrefixes' in response:
        kb_folders = []
        for prefix in response['CommonPrefixes']:
            folder_name = prefix['Prefix'].rstrip('/')
            if folder_name.startswith('knowledge-base-'):
                kb_folders.append(folder_name)
        
        if kb_folders:
            print(f"[INFO] Found {len(kb_folders)} existing knowledge base folder(s):")
            # Show last 5 folders
            for folder in sorted(kb_folders)[-5:]:
                print(f"  - s3://{BUCKET_NAME}/{folder}/")
            print(f"\nCreating NEW folder: {KB_FOLDER}")
            print("This will NOT affect existing folders or their ingestion jobs")
        else:
            print("[INFO] No existing knowledge base folders found")
            print(f"This will be the first folder: {KB_FOLDER}")
    else:
        print("[INFO] No existing folders found")
        print(f"This will be the first folder: {KB_FOLDER}")
        
except Exception as e:
    print(f"[WARNING] Could not check existing folders: {e}")
    print(f"Proceeding with new folder: {KB_FOLDER}")

In [ ]:
# List existing knowledge base versions
print("Checking for existing versions...")
print("-"*80)

try:
    response = s3_client.list_objects_v2(
        Bucket=BUCKET_NAME,
        Prefix="knowledge-base/",
        Delimiter='/'
    )
    
    if 'CommonPrefixes' in response:
        versions = []
        for prefix in response['CommonPrefixes']:
            version = prefix['Prefix'].replace('knowledge-base/', '').rstrip('/')
            versions.append(version)
        
        if versions:
            print(f"[INFO] Found {len(versions)} existing version(s):")
            for v in sorted(versions):
                print(f"  - {v}")
            print(f"\nCreating new version: v_{timestamp}")
        else:
            print("[INFO] No existing versions found")
            print(f"This will be the first version: v_{timestamp}")
    else:
        print("[INFO] No existing versions found")
        print(f"This will be the first version: v_{timestamp}")
        
except Exception as e:
    print(f"[WARNING] Could not check existing versions: {e}")
    print(f"Proceeding with new version: v_{timestamp}")

In [ ]:
# Check bucket exists and is accessible
try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"[OK] Bucket '{BUCKET_NAME}' exists and is accessible")
    
    # Get bucket location
    location = s3_client.get_bucket_location(Bucket=BUCKET_NAME)
    bucket_region = location['LocationConstraint'] or 'us-east-1'
    print(f"[OK] Bucket region: {bucket_region}")
    
    if bucket_region != AWS_REGION:
        print(f"[WARNING] Bucket region ({bucket_region}) differs from configured region ({AWS_REGION})")
        
except Exception as e:
    print(f"[FAILED] Cannot access bucket: {e}")
    print("\nPlease ensure:")
    print("1. Bucket name is correct")
    print("2. AWS credentials are valid")
    print("3. IAM permissions include s3:GetBucket* and s3:PutObject")

---

## 3. Load Knowledge Base Chunks

Load the chunks generated in Notebook 01.

In [ ]:
# Load chunks from JSON file
chunks_file = '../data/knowledge_base_chunks.json'

print(f"Loading chunks from: {chunks_file}")

try:
    with open(chunks_file, 'r') as f:
        chunks = json.load(f)
    
    print(f"[OK] Loaded {len(chunks)} chunks")
    
    # Count by source
    policy_chunks = [c for c in chunks if c['source'] != 'faq']
    faq_chunks = [c for c in chunks if c['source'] == 'faq']
    
    print(f"  Policy chunks: {len(policy_chunks)}")
    print(f"  FAQ chunks: {len(faq_chunks)}")
    
    # Show sample chunk
    print("\nSample chunk structure:")
    print("-"*80)
    print(json.dumps(chunks[0], indent=2))
    
except FileNotFoundError:
    print(f"[FAILED] File not found: {chunks_file}")
    print("Please run Notebook 01 first to generate the chunks")
except Exception as e:
    print(f"[FAILED] Error loading chunks: {e}")

---

## 4. Prepare Files for Upload

Create individual text files for each chunk.

In [ ]:
# Create temporary directory for files
temp_dir = Path('temp_kb_upload')
temp_dir.mkdir(exist_ok=True)

# Create subdirectories
(temp_dir / 'policies').mkdir(exist_ok=True)
(temp_dir / 'faqs').mkdir(exist_ok=True)

print("Creating text files for upload...")
print("-"*80)

files_created = []

# Process policy chunks
for chunk in policy_chunks:
    filename = f"{chunk['chunk_id']}.txt"
    filepath = temp_dir / 'policies' / filename
    
    with open(filepath, 'w') as f:
        f.write(chunk['text'])
    
    files_created.append(('policies', filename))

# Process FAQ chunks
for chunk in faq_chunks:
    filename = f"{chunk['chunk_id']}.txt"
    filepath = temp_dir / 'faqs' / filename
    
    with open(filepath, 'w') as f:
        f.write(chunk['text'])
    
    files_created.append(('faqs', filename))

print(f"[OK] Created {len(files_created)} text files")
print(f"  Policies: {len(policy_chunks)} files in temp_kb_upload/policies/")
print(f"  FAQs: {len(faq_chunks)} files in temp_kb_upload/faqs/")

# Show sample files
print("\nSample files created:")
for subfolder in ['policies', 'faqs']:
    sample_files = list((temp_dir / subfolder).glob('*.txt'))[:3]
    print(f"\n{subfolder.upper()}:")
    for f in sample_files:
        print(f"  - {f.name}")

---

## 5. Upload to S3

Upload all files to the S3 bucket with progress tracking.

In [ ]:
print("Uploading files to S3...")
print("="*80)

uploaded = 0
failed = 0
upload_details = []

for subfolder, filename in files_created:
    local_path = temp_dir / subfolder / filename
    s3_key = f"{KB_FOLDER}/{subfolder}/{filename}"  # Use KB_FOLDER
    
    try:
        s3_client.upload_file(
            str(local_path),
            BUCKET_NAME,
            s3_key,
            ExtraArgs={'ContentType': 'text/plain'}
        )
        uploaded += 1
        upload_details.append({
            'file': filename,
            'folder': subfolder,
            's3_key': s3_key,
            'status': 'success'
        })
        
        # Progress indicator every 10 files
        if uploaded % 10 == 0:
            print(f"  Progress: {uploaded}/{len(files_created)} files uploaded...")
            
    except Exception as e:
        failed += 1
        print(f"  [FAILED] Could not upload {filename}: {e}")
        upload_details.append({
            'file': filename,
            'folder': subfolder,
            's3_key': s3_key,
            'status': 'failed',
            'error': str(e)
        })

print(f"\n[OK] Upload complete!")
print(f"  Uploaded: {uploaded}/{len(files_created)} files")
print(f"  S3 Location: s3://{BUCKET_NAME}/{KB_FOLDER}/")
print(f"  Failed: {failed}")

if failed > 0:
    print("\nFailed uploads:")
    for detail in upload_details:
        if detail['status'] == 'failed':
            print(f"  - {detail['file']}: {detail['error']}")

---

## 6. Create and Upload Manifest

Generate metadata manifest for the knowledge base.

In [ ]:
# Create manifest with metadata
manifest = {
    "knowledge_base_name": "ecommerce-kb",
    "description": "E-commerce customer support knowledge base",
    "folder_name": KB_FOLDER,
    "total_documents": len(files_created),
    "sources": {
        "policies": len(policy_chunks),
        "faqs": len(faq_chunks)
    },
    "s3_location": f"s3://{BUCKET_NAME}/{KB_FOLDER}/",
    "created_at": datetime.now().isoformat(),
    "upload_summary": {
        "total_files": len(files_created),
        "uploaded": uploaded,
        "failed": failed
    }
}

# Save manifest locally
manifest_path = temp_dir / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print("Manifest created:")
print("="*80)
print(json.dumps(manifest, indent=2))

# Upload manifest to S3
try:
    s3_client.upload_file(
        str(manifest_path),
        BUCKET_NAME,
        f"{KB_FOLDER}/manifest.json",
        ExtraArgs={'ContentType': 'application/json'}
    )
    print("\n[OK] Manifest uploaded to S3")
except Exception as e:
    print(f"\n[FAILED] Could not upload manifest: {e}")

---

## 7. Verify S3 Upload

List files in S3 to confirm upload.

In [ ]:
print("Verifying S3 upload...")
print("="*80)

# List objects in S3
try:
    response = s3_client.list_objects_v2(
        Bucket=BUCKET_NAME,
        Prefix=KB_FOLDER
    )
    
    if 'Contents' in response:
        objects = response['Contents']
        print(f"[OK] Found {len(objects)} objects in S3")
        
        # Count by folder
        policies_count = sum(1 for obj in objects if 'policies/' in obj['Key'])
        faqs_count = sum(1 for obj in objects if 'faqs/' in obj['Key'])
        
        print(f"\nBreakdown:")
        print(f"  Policies: {policies_count} files")
        print(f"  FAQs: {faqs_count} files")
        print(f"  Other: {len(objects) - policies_count - faqs_count} files (manifest, etc.)")
        
        # Show sample objects
        print("\nSample S3 objects:")
        for obj in objects[:5]:
            size_kb = obj['Size'] / 1024
            print(f"  - {obj['Key']} ({size_kb:.2f} KB)")
    else:
        print("[WARNING] No objects found in S3")
        
except Exception as e:
    print(f"[FAILED] Could not list S3 objects: {e}")

---

## 8. Cleanup Temporary Files

Remove local temporary files after successful upload.

In [ ]:
import shutil

# Only cleanup if upload was successful
if failed == 0:
    try:
        shutil.rmtree(temp_dir)
        print(f"[OK] Cleaned up temporary directory: {temp_dir}")
    except Exception as e:
        print(f"[WARNING] Could not remove temp directory: {e}")
else:
    print(f"[INFO] Keeping temporary files due to failed uploads")
    print(f"  Location: {temp_dir}")
    print("  Review failed files and re-run upload if needed")

---

## 9. Next Steps for Bedrock Knowledge Base

Instructions for creating the Knowledge Base in AWS.

In [ ]:
from IPython.display import Markdown, display

next_steps = f"""
## SUCCESS! Knowledge Base Uploaded to S3

### S3 Location
```
s3://{BUCKET_NAME}/{KB_FOLDER}/
```

### File Structure
```
s3://{BUCKET_NAME}/{KB_FOLDER}/
├── policies/     ({len(policy_chunks)} files)
├── faqs/         ({len(faq_chunks)} files)
└── manifest.json (metadata)
```

### Important: Unique Folder Per Upload

Each upload creates a **separate timestamped folder**:
- `knowledge-base-20260117_103042/`  ← Upload #1
- `knowledge-base-20260117_143521/`  ← Upload #2
- `knowledge-base-20260118_091234/`  ← Upload #3

**Benefits:**
- No duplicate data ingestion
- Each Knowledge Base data source points to ONE folder
- Safe to update without affecting existing KB
- Version control and rollback capability

### Next Steps

#### 1. Create Bedrock Knowledge Base

Go to AWS Bedrock Console:
https://console.aws.amazon.com/bedrock/

#### 2. Navigate to Knowledge Bases
- Click "Knowledge bases" in left sidebar
- Click "Create knowledge base"

#### 3. Configure Knowledge Base

**Basic Information:**
- Name: `ecommerce-kb-{timestamp}` (or any name you prefer)
- Description: `E-commerce customer support KB - Version {timestamp}`

**Data Source:**
- Type: Amazon S3
- S3 URI: `s3://{BUCKET_NAME}/{KB_FOLDER}/`  ← **USE THIS EXACT PATH**
- Data deletion policy: DELETE (removes from vector DB when deleted from S3)

**IMPORTANT:** Point to **THIS specific folder** to avoid duplicate ingestion!

**Chunking Strategy:**
- Strategy: No chunking (we pre-chunked in Notebook 01)
- Each file is already an optimal chunk

**Embeddings:**
- Model: `amazon.titan-embed-text-v2:0`
- Dimensions: 1024

**Vector Store:**
- Option 1: OpenSearch Serverless (recommended, fully managed)
- Option 2: Amazon Aurora (if you have existing Aurora cluster)

#### 4. Create and Wait for Ingestion
- Click "Create"
- Wait for status to change to "Active" (2-5 minutes)
- Ingestion job will run automatically
- Monitor progress in "Data sources" tab

#### 5. Get Knowledge Base ID
- Copy the Knowledge Base ID (e.g., `XVIAYNVNB2`)
- Add to your `.env` file:
  ```bash
  KNOWLEDGE_BASE_ID=YOUR_KB_ID_HERE
  ```

#### 6. Test in Notebook 02
- Open `02_bedrock_knowledge_base.ipynb`
- Run cells to test retrieval
- Verify documents are indexed correctly

### Updating Knowledge Base

When you want to update with new data:

1. **Run this notebook again** - Creates a NEW folder with timestamp
2. **Create a NEW data source** in your existing KB pointing to the new folder
3. **OR** Update existing data source S3 URI to point to new folder
4. **Trigger ingestion** - Only new folder data gets indexed
5. **Old folder remains intact** - Can revert if needed

### Troubleshooting

**Issue: Duplicate Data in Ingestion**
- Make sure S3 URI points to the SPECIFIC timestamped folder
- NOT: `s3://{BUCKET_NAME}/` (would ingest all folders!)
- YES: `s3://{BUCKET_NAME}/{KB_FOLDER}/` (only this version)

**Issue: Ingestion Failed**
- Check S3 bucket permissions
- Verify IAM role has Bedrock and S3 access
- Check file encoding (should be UTF-8)

**Issue: No Documents Indexed**
- Verify S3 URI is exactly: `s3://{BUCKET_NAME}/{KB_FOLDER}/`
- Check data source is synced
- Re-run ingestion job

**Issue: Permission Errors**
- IAM role needs:
  - `s3:GetObject` on bucket
  - `bedrock:InvokeModel` for embeddings
  - OpenSearch access (if using OSS)
"""

display(Markdown(next_steps))

---

## Summary

In this notebook, we:

1. Loaded chunked knowledge base from Notebook 01
2. Created individual text files for each chunk
3. Organized files into policies/ and faqs/ folders
4. Uploaded all files to S3 bucket
5. Generated and uploaded metadata manifest
6. Verified upload success
7. Cleaned up temporary files
8. Provided instructions for Bedrock KB creation

**Key Takeaways:**

- S3 is the data source for Bedrock Knowledge Bases
- Pre-chunking gives us control over chunk quality
- Organized folder structure helps with management
- Manifest files provide useful metadata
- Upload can be automated for CI/CD pipelines

**Next**: Proceed to Notebook 02 to set up and test the Bedrock Knowledge Base!